In [1]:
# Install required packages if necessary:
# pip install numpy pandas matplotlib seaborn scikit-learn statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.datasets import make_regression, fetch_california_housing, load_wine
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler

# Reproducibility
np.random.seed(42)

# ============================================================
# 1. SIMPLE LINEAR REGRESSION
# ============================================================
print("\n" + "="*60)
print("1. SIMPLE LINEAR REGRESSION")
print("="*60)

# Generate synthetic data
X_simple, y_simple = make_regression(
    n_samples=150, n_features=1, noise=10, random_state=42
)
X_simple = X_simple.reshape(-1, 1)

# Split data
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

# Model
simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)
y_pred_s = simple_model.predict(X_test_s)

# Results
print(f"Intercept: {simple_model.intercept_:.4f}")
print(f"Coefficient: {simple_model.coef_[0]:.4f}")
print(f"Regression equation: y = {simple_model.intercept_:.4f} + ({simple_model.coef_[0]:.4f})x")

# Plots
plt.figure(figsize=(7, 5))
plt.scatter(X_test_s, y_test_s, color="blue", label="Actual")
plt.plot(X_test_s, y_pred_s, color="red", linewidth=2, label="Regression line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

# ============================================================
# 2. MULTIPLE LINEAR REGRESSION
# ============================================================
print("\n" + "="*60)
print("2. MULTIPLE LINEAR REGRESSION")
print("="*60)

# Load dataset
housing = fetch_california_housing()
X_multiple = pd.DataFrame(housing.data, columns=housing.feature_names)
y_multiple = housing.target

# Split data
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multiple, y_multiple, test_size=0.2, random_state=42
)

# Model
multiple_model = LinearRegression()
multiple_model.fit(X_train_m, y_train_m)
y_pred_m = multiple_model.predict(X_test_m)

# Results
print("\nIntercept:", multiple_model.intercept_)
print("\nFeature coefficients:")
coef_df = pd.DataFrame({
    "Feature": X_multiple.columns,
    "Coefficient": multiple_model.coef_
})
print(coef_df)

# Validation
mae = mean_absolute_error(y_test_m, y_pred_m)
mse = mean_squared_error(y_test_m, y_pred_m)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_m, y_pred_m)
print(f"\nMAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

# ============================================================
# 3. LOGISTIC REGRESSION (GLM)
# ============================================================
print("\n" + "="*60)
print("3. LOGISTIC REGRESSION (GLM)")
print("="*60)

# Load dataset (binary classification)
wine = load_wine()
X_glm = pd.DataFrame(wine.data, columns=wine.feature_names)
y_glm = (wine.target != 0).astype(int)  # Binary: class 0 vs rest

# Split data
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_glm, y_glm, test_size=0.2, random_state=42, stratify=y_glm
)

# Scale data
scaler = StandardScaler()
X_train_g = scaler.fit_transform(X_train_g)
X_test_g = scaler.transform(X_test_g)

# Model
glm_model = LogisticRegression(max_iter=1000, solver="liblinear")
glm_model.fit(X_train_g, y_train_g)
y_pred_g = glm_model.predict(X_test_g)
y_prob_g = glm_model.predict_proba(X_test_g)[:, 1]

# Results
accuracy = accuracy_score(y_test_g, y_pred_g)
precision = precision_score(y_test_g, y_pred_g)
recall = recall_score(y_test_g, y_pred_g)
f1 = f1_score(y_test_g, y_pred_g)
auc = roc_auc_score(y_test_g, y_prob_g)
print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, ROC-AUC: {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test_g, y_pred_g)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Class 0", "Class 1"], yticklabels=["Class 0", "Class 1"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ROC Curve (manual)
fpr, tpr, _ = roc_curve(y_test_g, y_prob_g)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# ============================================================
# FINAL MODEL COMPARISON
# ============================================================
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
comparison = pd.DataFrame({
    "Model": ["Simple Linear Regression", "Multiple Linear Regression", "Logistic Regression (GLM)"],
    "Dataset": ["Synthetic", "California Housing", "Wine"],
    "Problem Type": ["Continuous", "Continuous", "Binary Classification"],
    "Key Metrics": ["MAE, RMSE, R2", "MAE, RMSE, R2", "Accuracy, Precision, Recall, F1, ROC-AUC"]
})
print(comparison.to_string(index=False))


1. SIMPLE LINEAR REGRESSION
Intercept: 1.3805
Coefficient: 90.3346
Regression equation: y = 1.3805 + (90.3346)x


<Figure size 700x500 with 1 Axes>


2. MULTIPLE LINEAR REGRESSION

Intercept: -37.02327770606402

Feature coefficients:
      Feature  Coefficient
0      MedInc     0.448675
1    HouseAge     0.009724
2    AveRooms    -0.123323
3   AveBedrms     0.783145
4  Population    -0.000002
5    AveOccup    -0.003526
6    Latitude    -0.419792
7   Longitude    -0.433708

MAE: 0.5332, MSE: 0.5559, RMSE: 0.7456, R2: 0.5758

3. LOGISTIC REGRESSION (GLM)
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000, ROC-AUC: 1.0000


<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 1 Axes>


FINAL MODEL COMPARISON
                      Model             Dataset           Problem Type                               Key Metrics
   Simple Linear Regression           Synthetic             Continuous                             MAE, RMSE, R2
 Multiple Linear Regression  California Housing             Continuous                             MAE, RMSE, R2
  Logistic Regression (GLM)                Wine  Binary Classification  Accuracy, Precision, Recall, F1, ROC-AUC
